In [21]:
import warnings
warnings.filterwarnings("ignore")

import owlrl
from rdflib import Dataset
from pathlib import Path

g = Dataset()
#g.parse("../data/Capus_LHommeBicycle_1893/Capus_LHommeBicycle_1893.ttl", format="turtle")
#g.parse("../data/Epheyre_LeMicrobeDuProfesseurBakermannRecitDesTempsFuturs_1890/Epheyre_LeMicrobeDuProfesseurBakermannRecitDesTempsFuturs_1890.ttl", format="turtle")
#g.parse("../data/Willy_UneInvention_1895/Willy_UneInvention_1895.ttl", format="turtle")

data_dir = Path("../data")

for ttl in sorted(data_dir.glob("*/*.ttl")):
    g.parse(ttl, format="turtle")

print(f"{len(list(data_dir.glob('*/*.ttl')))} fichiers chargés, {len(g)} triplets")

g.parse("../src/SFonto.ttl", format="turtle")

# Raisonnement
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)


249 fichiers chargés, 6166 triplets


In [22]:
prefix = """

PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/> 
PREFIX dc: <http://purl.org/dc/elements/1.1/> 
PREFIX dcterms: <http://purl.org/dc/terms/> 
PREFIX dlpext: <http://www.ontologydesignpatterns.org/ont/dlp/ExtendedDnS.owl#> 
PREFIX dlpfun: <http://www.ontologydesignpatterns.org/ont/dlp/FunctionalParticipation.owl#> 
PREFIX dlpspa: <http://www.ontologydesignpatterns.org/ont/dlp/SpatialRelations.owl#> 
PREFIX dlptem: <http://www.ontologydesignpatterns.org/ont/dlp/TemporalRelations.owl#> 
PREFIX dolce: <http://www.ontologydesignpatterns.org/ont/dlp/DOLCE-Lite.owl#> 
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#> 
PREFIX gc: <https://w3id.org/golem/ontology#> 
PREFIX lrm: <http://iflastandards.info/ns/lrm/lrmoo/> 
PREFIX owl: <http://www.w3.org/2002/07/owl#> 
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> 
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
PREFIX schema: <https://schema.org/> 
PREFIX sf: <https://w3id.org/ontosf/ontology#> 
PREFIX vann: <http://purl.org/vocab/vann/> 
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> 

"""

In [29]:
query = prefix + """
SELECT DISTINCT ?feature WHERE {
  ?feature a gc:G18_Textual_Feature .
  FILTER(?feature != sf:Feature_science_fiction_genre)
  FILTER EXISTS {
    ?work a lrm:F1_Work ; 
          gc:GP0_has_feature ?feature .
    ?expr a lrm:F2_Expression ;
          lrm:R3i_realises ?work ;
          dcterms:issued ?date .
    FILTER(xsd:integer(str(?date)) < 1920)
  }
  FILTER NOT EXISTS {
    ?work2 a lrm:F1_Work ; 
           gc:GP0_has_feature ?feature .
    ?expr a lrm:F2_Expression ;
          lrm:R3i_realises ?work2 ;
          dcterms:issued ?date2 .
    FILTER(xsd:integer(str(?date2)) >= 1920)
  }
}
"""
#  






In [ ]:
def display_results(results):
    rows = results 
    data = []
    for row in rows:
        data.append([str(val).split("#")[-1] if val else "(empty)" for val in row])
    
    if not data:
        print("(no result)")
        return
    
    vars_ = [str(v) for v in results.vars]
    col_widths = [max(len(h), max(len(r[i]) for r in data)) +5 for i, h in enumerate(vars_)]

    header = "  ".join(f"{h:<{col_widths[i]}}" for i, h in enumerate(vars_))
    separator = "-" * len(header)
    print(header)
    print(separator)
    for row in data:
        print("  ".join(f"{val:<{col_widths[i]}}" for i, val in enumerate(row)))

results = g.query(query)
display_results(results)

feature                        
-------------------------------
Feature_parodic_register       
Feature_poetry_register        
Feature_religious_register     


: 